In [3]:
import requests
import pandas as pd
import json
import time
from config import *
from datetime import datetime
from zoneinfo import ZoneInfo

API URL

In [4]:
italki = "https://api.italki.com/api/v2/teachers"
language = 'korean'

Define the next few functions

In [5]:
def get_data(page, min_price, max_price):
    headers_format = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Accept-Language': 'en-GB,en-US;q=0.9,en;q=0.8,zh-CN;q=0.7,zh;q=0.6',
        'Content-Length': '130',
        'Content-Type': 'application/json',
        'Origin': 'https://www.italki.com',
        'Priority': 'u=1, i',
        'Referer': 'https://www.italki.com/',
        'Sec-Ch-Ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
        'Sec-Ch-Ua-Platform': '?0',
        'Sec-Ch-Ua-Platform': "Windows",
        'Sec-Fetch-Dest': '',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site',
        'traceparent': traceparent,
        'User-Agent': personal_user_agent,
        'x-browser-key': personal_browser_key, # Found in the actual POST request
        'x-device': '10',                       # Common default, but check yours
        'x-locale': 'en',
        'x-token': personal_x_token    
    }

    payload = {
        "teach_language": {
            "language": language,
            "max_price": max_price,
            "min_price": min_price,
        },
        "page": page,
        "page_size": 20,
        "user_timezone": "Asia/Singapore"
    }

    response = requests.post(url = italki, headers = headers_format, json=payload)
    time.sleep(2)
    dictionary = response.json()
    return dictionary

In [6]:
def join_data(new_data, filename):
    with open(filename, 'r') as file:
        existing = json.load(file)
    
    existing_tuples = {tuple(sorted(d.items())) for d in existing}
    new_no_duplicates = []
    seen = set()
    for d in new_data:
        d_tuple = tuple(sorted(d.items()))

        if d_tuple not in seen:
            new_no_duplicates.append(d)
            seen.add(d_tuple)

    new_to_join = [
        d for d in new_no_duplicates
        if tuple(sorted(d.items())) not in existing_tuples
    ]

    combined = existing + new_to_join
    with open(filename, 'w') as file:
        json.dump(combined, file, indent=4)
    return len(new_to_join)

In [7]:
def override_data(new_data, filename):
    with open(filename, 'w') as file:
        json.dump(new_data, file, indent=4)

Appending new information to old JSONs

In [8]:
def user_course_info(teachers):
    user_course_info_list = []
    for teacher in teachers:
        user_course_info = teacher['user_info'] | teacher['course_info']
        del user_course_info['avatar_file_name']
        del user_course_info['is_online']
        del user_course_info['last_login_time']
        del user_course_info['trial_description']
        user_course_info_list.append(user_course_info)
    return user_course_info_list

    # user_course_info
    # df = pd.DataFrame(user_course_info_list)


In [9]:
def teacher_stats(teachers):
    teacher_stats_list = []
    for teacher in teachers:
        test_stat = {}
        test_stat['user_id'] = teacher['user_info']['user_id']
        test_stat['finished_session'] = teacher['teacher_statistics']['finished_session']
        test_stat['response_rate'] = teacher['teacher_statistics']['response_rate']
        test_stat['attendance_rate'] = teacher['teacher_statistics']['attendance_rate']
        teacher_stats_list.append(test_stat)
    return teacher_stats_list

In [10]:
def pro_course_prices(teachers):
    master_price_list = []
    pro_course_detail_list = []
    for teacher in teachers:
        for course in teacher['pro_course_detail']:
            for prices in course['price_list']:
                master_price_list.append(prices)
            course_no_price = course.copy()
            del course_no_price['price_list']
            del course_no_price['description']
            del course_no_price['level_lower_limit']
            del course_no_price['level_up_limit']
            del course_no_price['course_category']
            del course_no_price['course_tags']
            del course_no_price['create_time']
            pro_course_detail_list.append(course_no_price)
    return master_price_list, pro_course_detail_list


In [11]:
with open('also_speaks_reference.json', 'r') as file:
    also_speak_index_list = json.load(file)
def check_language(language):
    if language in also_speak_index_list:
        return also_speak_index_list.index(language)
    else:
        also_speak_index_list.append(language)
        return also_speak_index_list.index(language)

In [12]:
def also_speak(teachers):
    also_speak_list = []
    for teacher in teachers:
        user_id = teacher['user_info']['user_id']
        for language in teacher['teacher_info']['also_speak']:
            current_language = language['language']
            index = check_language(current_language)
            id_language = {}
            id_language['user_id'] = user_id
            id_language['language'] = index
            # id_language['map'] = index
            also_speak_list.append(id_language)
    return also_speak_list

Record Book\
Takes a dictionary with the keys, min_price, max_price, number of records and datetime and append to a list

In [13]:
date_time_now = str(datetime.now(ZoneInfo('Asia/Singapore')))
date_time_now

'2026-06-02 13:58:51.647591+08:00'

In [14]:
def record_book(min_price, max_price, num_records, records_added):
    record_book_list = []
    date_time_now = str(datetime.now(ZoneInfo('Asia/Singapore')))
    records = {'min_price': min_price,
                'max_price': max_price,
                'total_records_called': num_records,
                'records_added': records_added,
                'date_time': date_time_now,
                'language': language}
    record_book_list.append(records)
    return record_book_list

Check number of entries per price range

In [40]:
min_price = 5000
max_price = min_price + 7099

In [41]:
# Change price here
def check_entries(min_price, max_price):
    dictionary = get_data(1, min_price, max_price)
    paging = dictionary['paging']
    return paging

page = check_entries(min_price, max_price)
print(page)
if int(page['total']) % 20 == 0:
    range_until = (int(page['total']) // 20) + 1
else:
    range_until = (int(page['total']) // 20) + 2
print(range_until)

{'page': 1, 'page_size': 20, 'total': 33, 'has_next': 1}
3


In [42]:
print(min_price, max_price, range_until, language)

5000 12099 3 korean


Loop through pages

In [43]:
user_course_info_cycle = []
teacher_stats_cycle = []
pro_course_cycle = []
price_list_cycle = []
also_speak_list_cycle = []

has_next = 1
page = 1
print(min_price, max_price)
print('page 1')
while has_next == 1:
    api_call = get_data(page, min_price, max_price)
    teachers = api_call['data']
    user_course_info_cycle += user_course_info(teachers)
    teacher_stats_cycle += teacher_stats(teachers)
    price_list_buffer, pro_course_buffer = pro_course_prices(teachers)
    pro_course_cycle += pro_course_buffer
    price_list_cycle += price_list_buffer
    also_speak_list_cycle += also_speak(teachers)
    
    page += 1
    has_next = api_call['paging']['has_next']
    print('page', page)

records_added = join_data(user_course_info_cycle, 'user_course_info.json')
join_data(teacher_stats_cycle, 'teacher_stats.json')
join_data(pro_course_cycle, 'pro_course.json')
join_data(price_list_cycle, 'price_list.json')
join_data(also_speak_list_cycle, 'also_speaks.json')
override_data(also_speak_index_list, 'also_speaks_reference.json')

records_cycle = record_book(min_price, max_price, api_call['paging']['total'], records_added)

join_data(records_cycle, 'records.json')
# df = pd.DataFrame(user_course_info_list)


5000 12099
page 1
page 2
page 3


1

In [24]:
with open('also_speaks.json', 'r') as file:
    also_speak_list = json.load(file)

with open('also_speaks_reference.json', 'r') as file:
    also_speaks_index_list = json.load(file)

df_also_speak_list = pd.DataFrame(also_speak_list)
df_languages = pd.DataFrame(also_speak_index_list, columns=['other languages'])
result = pd.merge(df_also_speak_list, df_languages, left_on='language', right_index=True, how="left")
result

,user_id,language,other languages
0,5467830,0,spanish
1,5467830,1,arabic
2,5467830,2,arabic(maghrebi)
3,11401921,3,filipino(tagalog)
4,11401921,4,japanese
...,...,...,...
320,30589958,1,arabic
321,30589958,5,other
322,30408613,11,french
323,30408613,4,japanese


In [44]:
with open('user_course_info.json', 'r') as file:
    user_course_info_check = json.load(file)

df_user_course_info_check = pd.DataFrame(user_course_info_check)

In [45]:
df_user_course_info_check[df_user_course_info_check['user_id'].duplicated(keep=False)]

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,5467830,⭐Learn with Kim/Kin,1,0,CN,KR,CN00001,Shanghai,KR00001,Seoul,Asia/Shanghai,2,0,1988,500,562,1
5,7023372,✨Galya✨,1,0,RU,RU,RU00006,St. Petersburg,RU00006,St. Petersburg,Europe/Moscow,2,0,1500,500,321,1
75,10654806,Kwabena,1,0,GH,GH,GH00001,Accra,GH00001,Accra,Atlantic/Reykjavik,2,0,550,600,116,0
76,28084220,Eileen,1,0,SG,JP,SG00000,Other,JP00000,Other,Asia/Tokyo,2,0,500,680,29,0
93,10599672,Naledi (Corporate) ✨,1,0,ZA,ZA,ZA00106,Klerksdorp,ZA00251,Sol Plaatje,Africa/Johannesburg,2,0,500,695,160,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11096,1438812,Meishan,0,1,CN,KR,CN00091,Changchun,ZZ00000,Other,Asia/Shanghai,2,0,500,3500,44,0
11098,9484347,Mira,1,0,TR,TR,TR00000,Other,TR00000,Other,Europe/Istanbul,2,0,1000,3000,351,0
11101,19707974,Jay,1,0,KR,FR,KR00001,Seoul,FR00000,Other,Europe/Paris,2,0,1500,3500,18,1
11135,26420047,RYAN LEE,1,0,KR,KR,KR00016,Daegu,KR00016,Daegu,Asia/Seoul,2,0,800,4000,22,0


In [46]:
df_user_course_info_check['user_id'].nunique()

10470

In [47]:
df_user_course_info_check['user_id'].count()

np.int64(11169)

In [48]:
df_user_course_info_check

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,5467830,⭐Learn with Kim/Kin,1,0,CN,KR,CN00001,Shanghai,KR00001,Seoul,Asia/Shanghai,2,0,1988,500,562,1
1,11401921,Teacher Emee,1,0,PH,PH,PH00000,Other,PH00000,Other,Asia/Manila,2,0,500,500,186,1
2,9627036,Shyam Syangtan,1,0,IN,IN,IN00771,Sonipat,IN00231,Delhi,Asia/Kolkata,2,0,700,500,1366,1
3,5787377,Kimi(y)a 👩🏻🎓❤️,1,0,IR,IT,IR00016,Tabriz,IT00016,Turin,Asia/Tehran,2,0,799,500,384,1
4,9934037,Anne Rola,1,0,PH,PH,PH00000,Other,PH00236,Tacloban,Asia/Shanghai,2,0,500,500,115,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11164,8392625,Yunhee Jung,1,0,US,US,US00346,East Los Angeles,US00346,East Los Angeles,America/Los_Angeles,2,0,1500,5000,0,1
11165,9170372,Hera 최해라,1,1,KR,GB,KR00166,Gunpo,GB00001,London,Europe/London,2,0,1500,5000,291,1
11166,23372570,MyKoreanTeacherDan,0,1,NZ,NZ,NZ00000,Other,NZ00001,Auckland,Pacific/Auckland,3,0,3500,5000,0,0
11167,8296973,Leeji 리지(Ji Eun Lee),0,1,KR,DE,KR00001,Seoul,DE00101,Karlsruhe,Europe/Berlin,2,0,2000,5000,0,1


In [49]:
with open('pro_course.json', 'r') as file:
    pro_course_to_df = json.load(file)

df_pro_course = pd.DataFrame(pro_course_to_df)

In [50]:
df_pro_course[df_pro_course['language'] == language].groupby('teacher_id')['session_price'].mean()

teacher_id
132815      1860.000000
815733      3233.333333
901015      2433.333333
948628      2666.666667
961354      1700.000000
               ...     
32098917    2550.000000
32099223    1400.000000
32101494    1500.000000
32102519    1000.000000
32105495    3200.000000
Name: session_price, Length: 538, dtype: float64

In [51]:
df_pro_course[df_pro_course['language'] == language].groupby('teacher_id').count()

,id,language,title,session_price,student_count,session_count,has_package
teacher_id,,,,,,,
132815,5,5,5,5,5,5,5
815733,3,3,3,3,3,3,3
901015,3,3,3,3,3,3,3
948628,6,6,6,6,6,6,6
961354,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...
32098917,4,4,4,4,4,4,4
32099223,1,1,1,1,1,1,1
32101494,1,1,1,1,1,1,1
